In [1]:
import os
import sys
import itertools
import subprocess
import nibabel as nib

sys.path.append('/host/verges/tank/data/daniel/00_commonUtils/00_code/genUtils/')
import gen, t1
import bids_naming as names
import dataChecks as check

In [ ]:
# From native surface, sphere of native surface, map onto target sphere
def do_surf_resample(surf_in:str, surf_out:str, sphere_in:str, sphere_target:str, method="BARYCENTRIC"):
    cmd = f"wb_command -surface-resample {surf_in} {sphere_in} {sphere_target} {method} {surf_out}"
    subprocess.run(cmd, shell=True, check=True)
    if not os.path.exists(surf_out):
        print(f"Error: {surf_out} was not created.")
        return None
    else:
        return surf_out

# parameters

study_dicts = [ # All surface paths listed in format [L, R]
    { # Bigbrain
        'studyName': 'Bigbrain',
        'studyDescrip': 'histology',
        'dir_root': '/host/verges/tank/data/daniel/04_inVivoHistology/data/bigbrain/sub-bigbrain/',
        'dir_anat': 'anat',
        'dir_surfs': 'surfs',
        'anat_volume': 'full16_100um_desc-optbal_space-histology.nii.gz', # NOTE. Surfaces must be in same space as volume
        'surf_ctx_midthickness_native': ["sub-bigbrain_hemi-L_space-hist_surf-hist-164k_label-midthickness.surf.gii",
                                         "sub-bigbrain_hemi-R_space-hist_surf-hist-164k_label-midthickness.surf.gii"],
        'surf_ctx_pial_native': ["sub-bigbrain_hemi-L_space-hist_surf-hist-164k_label-pial.surf.gii",
                                 "sub-bigbrain_hemi-R_space-hist_surf-hist-164k_label-pial.surf.gii"],
        'surf_ctx_white_native': ["sub-bigbrain_hemi-L_space-hist_surf-hist-164k_label-white.surf.gii",
                                  "sub-bigbrain_hemi-R_space-hist_surf-hist-164k_label-white.surf.gii"],
        'surf_hipp_inner_hu': ["sub-bigbrain_hemi-L_space-hist_den-0p5mm_label-hipp_inner.surf.gii",
                               "sub-bigbrain_hemi-R_space-hist_den-0p5mm_label-hipp_inner.surf.gii"], # 7262 vertices
        'surf_hipp_outer_hu': ["sub-bigbrain_hemi-L_space-hist_den-0p5mm_label-hipp_outer.surf.gii",
                               "sub-bigbrain_hemi-R_space-hist_den-0p5mm_label-hipp_outer.surf.gii"], # 7262 vertices
    }
]

hemis = ['L', 'R']

tpl_dir = "/host/verges/tank/data/daniel/04_inVivoHistology/data/bigbrain/sub-bigbrain/surfs/tpl/"
bb_nativeSurf_sphere_rot = ["lh.sphere_fsLR_rsled_like_BigBrain.sphere.reg.surf.gii",
                            "rh.sphere_fsLR_rsled_like_BigBrain.sphere.reg.surf.gii"]
tpl_164ksphere = ["lh.fsLR.sphere.surf.gii", 
                  "rh.fsLR.sphere.surf.gii"]
tpl_32kspheres = ["fsLR-32k.L.sphere.reg.surf.gii",
               "fsLR-32k.R.sphere.reg.surf.gii"]

for label, (i, hemi) in itertools.product(['surf_ctx_pial_native', 'surf_ctx_white_native'], enumerate(hemis)):
    surf_in_path = os.path.join(study_dicts[0]['dir_root'], study_dicts[0]['dir_surfs'], study_dicts[0][label][i])
    print(surf_in_path)

    # 1. Align native surface to fsLR-164k
    sphere_in_1 = os.path.join(tpl_dir, tpl_164ksphere[i])
    sphere_target_1 = os.path.join(tpl_dir, bb_nativeSurf_sphere_rot[i])
    surf_out_1 = surf_in_path.replace("surf-hist-164k", "surf-fsLR-164k")

    surf_out_1 = do_surf_resample(surf_in=surf_in_path, surf_out=surf_out_1,
                        sphere_in=sphere_in_1, sphere_target=sphere_target_1)

    print(f"\tfsLR-164k output:\n{subprocess.run('wb_command -surface-information ' + surf_out_1, shell=True, capture_output=True, text=True).stdout}\n")
    
    # 2. Downsample to fsLR-32k 
    sphere_in_2 = sphere_target_1
    sphere_target_2 = os.path.join(tpl_dir, tpl_32kspheres[i])
    surf_out_2 = surf_out_1.replace("surf-fsLR-164k", "surf-fsLR-32k")
    
    final_out_pth = do_surf_resample(surf_in=surf_out_1, surf_out=surf_out_2,
                        sphere_in=sphere_in_2, sphere_target=sphere_target_2)
    
    print(f"Saved {gen.fmt_file_size(final_out_pth)}: {final_out_pth}")
    print(f"\ fsLR-32k toutput:\n{subprocess.run('wb_command -surface-information ' + final_out_pth, shell=True, capture_output=True, text=True).stdout}\n{'-'*40}")

/host/verges/tank/data/daniel/04_inVivoHistology/data/bigbrain/sub-bigbrain/surfs/sub-bigbrain_hemi-L_space-hist_surf-hist-164k_label-pial.surf.gii
	fsLR-164k output:
Name: sub-bigbrain_hemi-L_space-hist_surf-fsLR-164k_label-pial.surf.gii
Type: Unknown
Number of Vertices: 163842
Number of Triangles: 327680
Bounds: (-64.0829, 5.38262, -70.0648, 78.0528, -36.1514, 54.6886)
Spacing:
    Mean: 0.939576
    Std Dev: 0.462452
    Minimum: 0.058818
    Maximum: 3.722288



Saved 709KB: /host/verges/tank/data/daniel/04_inVivoHistology/data/bigbrain/sub-bigbrain/surfs/sub-bigbrain_hemi-L_space-hist_surf-fsLR-32k_label-pial.surf.gii
\ fsLR-32k toutput:
Name: sub-bigbrain_hemi-L_space-hist_surf-fsLR-32k_label-pial.surf.gii
Type: Unknown
Number of Vertices: 32492
Number of Triangles: 64980
Bounds: (-63.9579, 4.87222, -70.0049, 77.8929, -36.0264, 54.5992)
Spacing:
    Mean: 1.966739
    Std Dev: 1.025326
    Minimum: 0.072134
    Maximum: 7.363103


----------------------------------------
/host/ve

In [ ]:
import stitchSurfs as stitch
import numpy as np
import vtk
import pyvista as pv


def load_template(template_path: str):
    class _RedirectUnpickler(pickle.Unpickler):
        def find_class(self, module, name):
            if module == "__main__":
                try:
                    import stitchSurfs as _local_mod
                    return getattr(_local_mod, name)
                except Exception:
                    return super().find_class(module, name)
            return super().find_class(module, name)

    with open(template_path, "rb") as f:
        template = _RedirectUnpickler(f).load()
    return template

def get_vtk_faces(faces):
    # VTK faces format: [3, v0,v1,v2, 3, v0,v1,v2, ...]
    n_faces = faces.shape[0]
    vtk_faces = np.empty((n_faces * 4,), dtype=np.uint32)  # 3 verts + 1 count per triangle
    vtk_faces[0::4] = 3  # Triangle marker
    vtk_faces[1::4] = faces[:, 0]
    vtk_faces[2::4] = faces[:, 1]  
    vtk_faces[3::4] = faces[:, 2]

    return vtk_faces

def show_mesh(vertices, faces, title: str = None, vertices_highlight: None | list = None):
    vtk_faces = get_vtk_faces(faces)  # assume this exists

    # Create PyVista mesh
    mesh = pv.PolyData(vertices, vtk_faces)
    plotter = pv.Plotter()
    
    if vertices_highlight is not None:
        labels = np.zeros(len(vertices), dtype=int)
        if type(vertices_highlight[0]) in [list, set]:
            # Assign integer label to each vertex (0=background, 1+=group id)
            n_groups = len(vertices_highlight)
            for group_id, group_verts in enumerate(vertices_highlight, 1):
                # print(f"[group {group_id}] len {len(group_verts)}, type {type(group_verts)}: {group_verts}")
                labels[list(group_verts)] = group_id  # set label for vertices in this group
            
            mesh["highlight_groups"] = labels
            
            # Create discrete colormap: gray background + one color per group
            colors = ["lightgray"] + [f"C{i}" for i in range(n_groups)]
            plotter.add_mesh(mesh, scalars="highlight_groups", cmap=colors, 
                            show_edges=False, 
                            show_scalar_bar = False,
                            interpolate_before_map=False)
        else: # assumes list of verteex indices
            colors = np.zeros(len(vertices))
            colors[vertices_highlight] = 1   # mark special vertices
            mesh["highlight"] = colors
            plotter.add_mesh(mesh, scalars="highlight", 
                             cmap=["lightgray", "red"], 
                             show_edges=False,
                             show_scalar_bar = False)
    else:
        plotter.add_mesh(mesh, cmap='tab20', show_edges=False, edge_color='black')
    
    if title is not None:
        plotter.add_title(title)
    
    plotter.show()
    return mesh


stitch_tpl = load_template("./overlap_stitch_template_JD_Jan2026.pkl")
keep_idx = stitch_tpl.keep_cortex_idx

# show what vertices are removed on surface
removed_verts = np.setdiff1d(np.arange(len(stitch_tpl.keep_cortex_idx)), keep_idx)
print(f"Total ctx vertices: {stitch_tpl.n_cortex}")
print(f"Retained vertices: {len(keep_idx)}", keep_idx)
print(f"Removed vertices: {len(removed_verts)}", removed_verts)

bb_fslr32_pial_L = "./sub-bigbrain_hemi-L_space-hist_surf-fsLR-32k_label-pial.surf.gii"
surf_data = nib.load(bb_fslr32_pial_L)
_ = show_mesh( surf_data.darrays[0].data, surf_data.darrays[1].data, vertices_highlight=removed_verts, title=f"BigBrain fsLR-32k-L pial\n'fsLR-32k' vertices removed for mTemp stitching")

bb_fslr32_pial_R = "./sub-bigbrain_hemi-R_space-hist_surf-fsLR-32k_label-pial.surf.gii"
surf_data = nib.load(bb_fslr32_pial_R)
_ = show_mesh(surf_data.darrays[0].data, surf_data.darrays[1].data, vertices_highlight=removed_verts, title=f"BigBrain fsLR-32k-R pial\n'fsLR-32k' vertices removed for mTemp stitching")